In [3]:
import os
import pandas as pd
from datetime import datetime
import pydicom
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import csv
import json

from utils.selection import *


# Source of RAW DICOMs
BERN_PATH='/nasdata/Berner_Daten/Other_Side_Bilddaten' 
# Project data path
ANALYSIS_PATH='/home/bule/berner_daten_analyse/data'

APP_DATA_PATH = '/home/bule/berner_daten_analyse/apps/app_manual_preselection'

-test selection pascal

In [5]:
with open(os.path.join(APP_DATA_PATH,'pascal_sequences.json')) as json_file:
    selection_pascal_dict = json.load(json_file)

rows = []
for patid, selection_dict in selection_pascal_dict.items():
    for modality, sequences in selection_dict.items():
        for sequence in sequences:
            study_id, sequence_id = sequence.split('/')
            rows.append({
                'Patient_ID': patid,
                'Study_ID': study_id,
                'Sequence_ID': sequence_id,
                'Modality': modality,
                'Labeller': 'Pascal'  
            })
selection_pascal_df = pd.DataFrame(rows)
selection_pascal_df['Sequence_path']=selection_pascal_df.apply(lambda x: os.path.join(BERN_PATH,x['Patient_ID'],x['Study_ID'],x['Sequence_ID']),axis=1)

selection_pascal_df.head()

dwi_t1_df_pascal_test=pd.DataFrame(columns=['Patient_ID','Study_ID','Sequence_ID','File_Name','Date','Sequence_ID_split,','Modality'])


###### BRING THE DF IN THE CORRECT FORMAT for the 2_0_  <------------------------------- SHOULD ALSO BE DONE FOR THE SELECTICON FROMN HAKIM ETC

dwi_t1_df_pascal_test = []  # 

for _, row in selection_pascal_df.iterrows():
    for dcm_ending in os.listdir(row['Sequence_path']):        
        dwi_t1_df_pascal_test.append({
            'Patient_ID': row['Patient_ID'],
            'Study_ID': row['Study_ID'],
            'Sequence_ID': row['Sequence_ID'],
            'File_Name': dcm_ending,
            'Full_Path': os.path.join(row['Sequence_path'], dcm_ending),
            'Modality': row['Modality'].upper()
        })

dwi_t1_df_pascal_test = pd.DataFrame(dwi_t1_df_pascal_test)
dwi_t1_df_pascal_test['Date'] = dwi_t1_df_pascal_test['Study_ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))
dwi_t1_df_pascal_test['Sequence_ID_split'] = dwi_t1_df_pascal_test['Sequence_ID'].apply(lambda x: split_at_anynumber_seq(x))
dwi_t1_df_pascal_test.to_csv(os.path.join(ANALYSIS_PATH, 'dwi_t1_df_pascal_test.csv'), index=False)

## Selection by people with domain knowledge

- hakim

In [11]:
with open(os.path.join(APP_DATA_PATH,'xbzu_sequences.json')) as json_file:
    selection_hakim_dict = json.load(json_file)

rows = []
for patid, selection_dict in selection_hakim_dict.items():
    for modality, sequences in selection_dict.items():
        for sequence in sequences:
            study_id, sequence_id = sequence.split('/')
            rows.append({
                'Patient_ID': patid,
                'Study_ID': study_id,
                'Sequence_ID': sequence_id,
                'Modality': modality,
                'Labeller': 'Hakim'  
            })
selection_hakim_df = pd.DataFrame(rows)
print(selection_hakim_df)

{'0090905555': {'dwi': [], 't1': [], 'other': ['20150207_3609180/Neuro 1 LD__1.3.12.2.1107.5.4.5.153179.30000015020710232628100000717']}, '0082789223': {'dwi': ['20161003_4120929/resolve_3scan_trace_tra_p3_160_TRACEW__1.3.12.2.1107.5.2.43.67081.2016100307523780997803261.0.0.0'], 't1': ['20160521_4001514/t1_se_r_tra_KM__1.3.12.2.1107.5.2.30.27272.20160521054116624612261.0.0.0'], 'other': []}}


In [ ]:
# TODO implement MERGE LOGIC for selection files -> output paths for each patient 1 dwi and 1 anatomical seq if the work is done by all  labellers..

- julian

In [17]:
with open(os.path.join(APP_DATA_PATH,'Julian_sequences.json')) as json_file:
    selection_julian_dict = json.load(json_file)

rows = []
for patid, selection_dict in selection_hakim_dict.items():
    for modality, sequences in selection_dict.items():
        for sequence in sequences:
            study_id, sequence_id = sequence.split('/')
            rows.append({
                'Patient_ID': patid,
                'Study_ID': study_id,
                'Sequence_ID': sequence_id,
                'Modality': modality,
                'Labeller': 'Julian'  
            })
selection_julian_df = pd.DataFrame(rows)
print(selection_julian_df)

   Patient_ID          Study_ID  \
0  0090905555  20150207_3609180   
1  0082789223  20161003_4120929   
2  0082789223  20160521_4001514   

                                         Sequence_ID Modality Labeller  
0  Neuro 1 LD__1.3.12.2.1107.5.4.5.153179.3000001...    other   Julian  
1  resolve_3scan_trace_tra_p3_160_TRACEW__1.3.12....      dwi   Julian  
2  t1_se_r_tra_KM__1.3.12.2.1107.5.2.30.27272.201...       t1   Julian  


- martin

In [15]:

with open(os.path.join(APP_DATA_PATH,'martin_sequences.json')) as json_file:
    selection_martin_dict = json.load(json_file)

rows = []
for patid, selection_dict in selection_martin_dict.items():
    for modality, sequences in selection_dict.items():
        for sequence in sequences:
            study_id, sequence_id = sequence.split('/')
            rows.append({
                'Patient_ID': patid,
                'Study_ID': study_id,
                'Sequence_ID': sequence_id,
                'Modality': modality,
                'Labeller': 'Martin'  
            })
selection_martin_df = pd.DataFrame(rows)
print(selection_martin_df)


   Patient_ID          Study_ID  \
0  0000076813  20170615_4362442   

                                         Sequence_ID Modality Labeller  
0  ep2d_diff_3scan_trace_p2__1.3.12.2.1107.5.2.30...      dwi   Martin  


- MERGE overlap 

In [20]:
merge_df =selection_martin_df.merge(selection_hakim_df , how='inner', on=['Patient_ID','Study_ID','Sequence_ID','Modality'])
merge_df

,Patient_ID,Study_ID,Sequence_ID,Modality,Labeller_x,Labeller_y


merge_df =selection_martin_df.merge(selection_hakim_df)

- save mergefile

In [ ]:
# df_selection.to_csv(os.path.join(ANALYSIS_PATH,'dwi_t1_df_3.csv'), index=False)